# Learning to Rank: Pointwise, Pairwise, and RankNet

The narrative companion to `learning_to_rank_pairwise.py`, which **owns every number** — this notebook
imports it and walks the topic section by section. The evaluation layer *measured* a ranking; here we
*learn* one, three ways: **pointwise** (regress score onto grade), **pairwise** (RankNet), and a
**listwise** preview. The rigorous backbone is that NDCG and MAP are piecewise-constant in the scores —
zero gradient almost everywhere — so they cannot be gradient-optimized directly, which is *why* the
smooth pairwise logistic surrogate exists.

The substrate is the capstone's complementary-view finance corpus: three retrieval legs (lexical /
dense / late-interaction) over disjoint token windows, so each leg recalls neighbors the others miss and
a learned combiner has genuine headroom over both each leg and reciprocal-rank fusion.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))
import numpy as np
import learning_to_rank_pairwise as ltr

corpus = ltr._corpus()
print("docs", corpus["n_docs"], "queries", corpus["n_queries"], "TOPK", ltr.TOPK)
print("train queries", len(corpus["train_q"]), "/ test queries", len(corpus["test_q"]))

## The substrate: three complementary leg scores as features, oracle-tertile grades

Each document gets a feature vector `x = [s_lex, s_dense, s_li]` — the three legs' per-document scores,
standardized within each query (the legs live on incompatible scales). Relevance grades come from the
same exact-MaxSim oracle the evaluation layer used: the top-K documents get grades {1, 2, 3} by global
oracle-score tertiles, everything else grade 0. By construction `{grade >= 1}` equals the binary truth
set — the nesting anchor.

In [ ]:
# the per-query standardized features and the graded labels
print("feats shape", corpus["feats"].shape, "(queries, docs, 3)")
q0 = corpus["test_q"][0]
print("grades for one query:", {int(d): int(g) for d, g in sorted(corpus["grades"][q0].items())})
print("nesting anchor {grade>=1} == truth set:",
      {d for d, g in corpus["grades"][q0].items() if g >= 1} == corpus["truth_sets"][q0])

## Pointwise: least squares to grades

The pointwise reduction regresses the linear score `s = w . x` onto the grade by ordinary least
squares. It is the global mean-squared-error minimizer — but it optimizes *calibration* (matching the
grade magnitude), not order.

In [ ]:
tr = corpus["train_q"]
X = np.vstack([corpus["feats"][q] for q in tr])
y = np.concatenate([corpus["y"][q] for q in tr])
w_point = ltr.pointwise_solve(X, y)
print("pointwise weights [lex, dense, li, intercept]:", np.round(w_point, 4))

## Pairwise: RankNet, and why its loss is convex

RankNet models each preference as a Bernoulli trial, $P(i \succ j) = \sigma(s_i - s_j)$, and minimizes
the pairwise cross-entropy summed over within-query preference pairs. For a linear scorer the loss is a
sum of softplus functions of a linear map — **convex in the weights** — so it has a single global
optimum reachable by L-BFGS-B with the analytic gradient. No stochastic gradient descent, no
learning-rate schedule, fully reproducible. The Hessian is a nonnegative-weighted sum of rank-1 outer
products, hence positive semidefinite — the algebraic witness of convexity.

In [ ]:
deltas = ltr.all_train_pairs(corpus, tr)
print("training preference pairs:", deltas.shape[0])
w_rank = ltr.fit_ranknet(deltas)
print("RankNet weights [lex, dense, li]:", np.round(w_rank, 4))

# convexity: the Hessian is PSD, and two different starts reach the same optimum
H = ltr.ranknet_hessian(w_rank, deltas)
print("min Hessian eigenvalue (>= 0 => convex):", round(float(np.linalg.eigvalsh(H).min()), 6))
w_alt = ltr.fit_ranknet(deltas, x0=np.array([2.0, -1.0, 3.0]))
print("two starts reach the same w*:", np.allclose(w_rank, w_alt, atol=1e-5))

## Why learning to rank exists: rank metrics are flat

NDCG and MAP are **piecewise-constant** in the scores. The ranking changes only when two scores cross,
so on any open region with no tie the metric is constant — gradient zero almost everywhere, with jump
discontinuities exactly at the swaps. They cannot be gradient-optimized; the smooth pairwise logistic
is a *surrogate*. We morph the weights from the weakest single leg toward the learned combiner and watch
NDCG climb as a **staircase** while the surrogate loss falls **smoothly**.

In [ ]:
q = ltr.pick_worked_query(corpus)
w_a = ltr._weakest_leg_direction(corpus, corpus["test_q"])
sweep = ltr.surrogate_sweep(corpus, corpus["test_q"], w_a, w_rank, n=240)
print("NDCG climbs in", sweep["n_distinct_ndcg"], "plateaus over", len(sweep["swaps"]), "swaps")
print("NDCG endpoints:", round(sweep["ndcg"][0], 4), "->", round(sweep["ndcg"][-1], 4))
print("smooth loss endpoints:", round(sweep["loss"][0], 1), "->", round(sweep["loss"][-1], 1))

## Ranking is not regression (H1)

A model that is optimal in pointwise mean-squared error can lose on NDCG to a pairwise model with worse
error — *order beats calibration*. The deterministic witness: two queries where least squares
(magnitude-weighted) and the pairwise loss (pair-count-weighted) pick opposite weight signs, plus a
query-level feature pointwise spends on calibration while pairwise's gradient along it is exactly zero.
On the real corpus the two are near-tied, so the witness is the rigorous anchor and the corpus number a
reported observation.

In [ ]:
h1 = ltr.constructed_ranking_vs_regression()
print("constructed witness:")
print("  pointwise: MSE", round(h1["mse_pointwise"], 4), " NDCG", round(h1["ndcg_pointwise"], 4))
print("  pairwise : MSE", round(h1["mse_pairwise"], 4), " NDCG", round(h1["ndcg_pairwise"], 4))
print("  lower MSE yet worse NDCG for pointwise:",
      h1["mse_pointwise"] < h1["mse_pairwise"] and h1["ndcg_pointwise"] < h1["ndcg_pairwise"])
print("on the real corpus (reported):", {k: (round(v, 4) if isinstance(v, float) else v)
                                          for k, v in ltr.h1_on_corpus(corpus).items()})

## The finance climax: learned fusion vs the legs and RRF (H2)

A RankNet trained on labeled preferences over the three legs is the *supervised* alternative to
reciprocal-rank fusion's fixed, unsupervised rule. On held-out test queries the learned ranker beats
every single leg and RRF — pinned to the run, with the gain over RRF reported honestly against its
confidence interval (one synthetic corpus, not a universal verdict).

In [ ]:
res = ltr.learned_vs_baselines(corpus)
print("held-out test recall@10 / NDCG@10:")
for name, m in res["test"].items():
    print(f"  {name:16s} recall={m['recall']:.4f}  ndcg={m['ndcg']:.4f}")
print("winner:", res["winner"], "| learned gain over RRF:", round(res["learned_gain_over_rrf"], 4))
ci = res["learned_recall_ci"]
print("learned recall 95% CI:", round(ci["ci_lo"], 4), "-", round(ci["ci_hi"], 4),
      f"(se {ci['se']:.4f}, n={ci['n']})")

## The lambda forces — the bridge to LambdaRank

RankNet's gradient factorizes into a per-document **lambda force** $\lambda_i = \sum_j \lambda_{ij}$ —
the net pull on each document from its preference pairs. On the worked query the single grade-3
document is under-ranked and feels the strongest upward force. LambdaRank scales each pairwise force by
the $|\Delta\text{NDCG}|$ that swapping the pair would cause — the named bridge this topic stops at.

In [ ]:
w = w_rank
lam = ltr.lambda_forces(w, corpus["feats"][q], corpus["y"][q])
order = ltr.ranking_from_w(corpus, q, w)
print(f"worked query {q} — grades:", {int(d): int(g) for d, g in corpus["grades"][q].items()})
print("rank | doc | grade | lambda force (negative = pulled up)")
for rank, d in enumerate(order[:ltr.TOPK], start=1):
    print(f"  {rank:2d} | {d:3d} | {corpus['grades'][q].get(d, 0):d} | {lam[d]:+.3f}")

## Closing

The pairwise logistic is a **surrogate**: minimizing it does not in general minimize NDCG or MAP — the
consistency gap that motivates LambdaRank's $\Delta\text{NDCG}$ reweighting. Convexity holds only for
the linear scorer; a deep RankNet is non-convex. The pairwise reduction is position-blind, the gap
listwise objectives (ListNet, RankGPT) address. Every number above is owned by the tested `.py`.